In [8]:
import pandas as pd 
import os 

results_path = "../Results/CXRAY"

df = pd.DataFrame()
models = sorted(os.listdir(results_path))
for folder in models:
    if os.path.isdir(os.path.join(results_path,folder)):
        df_temp = pd.read_csv(os.path.join(results_path,folder,"results.csv"))
        df_temp["model"] = folder
        
        for i, row in df_temp.iterrows():
            image = row["image"]
            
            if image.startswith('MCU'):
                df_temp.at[i, "dataset"] = "Montgomery"
            elif image.startswith('JPC'):
                df_temp.at[i, "dataset"] = "JSRT"
            elif image.startswith('CHN'):
                df_temp.at[i, "dataset"] = "Shenzhen"
            else:
                df_temp.at[i, "dataset"] = "Padchest"
            
        df = pd.concat([df,df_temp])

for dataset in df["dataset"].unique():
    df_dataset = df[df["dataset"] == dataset]
    for organ in df_dataset["organ"].unique():
        df_organ = df_dataset[df_dataset["organ"] == organ]
        
        print(f"Dataset: {dataset}, Organ: {organ}")
        
        metrics = ["dc", "hd"]
        for metric in metrics:
            print(f"{metric.upper()}:")
            for model in df_organ["model"].unique():
                df_model = df_organ[df_organ["model"] == model]
                mean_value = df_model[metric].mean()
                mean_value = round(mean_value, 3)
                std_value = df_model[metric].std()
                std_value = round(std_value, 3)
                print(f"  {model}: {mean_value:.3f} ± {std_value:.3f}")
            print()

# include nnUNet dataframe
path = "/home/ngaggion/Documents/Mask2Graph/Results/cxray_nnUNet_results.csv"
# has a column named dataset
df_nnunet = pd.read_csv(path)
df_nnunet["model"] = "nnUNet"

df = pd.concat([df,df_nnunet])

Dataset: Shenzhen, Organ: Lungs
DC:
  ChestXRay_10: 0.961 ± 0.015
  ChestXRay_10_NN: 0.963 ± 0.014
  ChestXRay_10_NN_no_raster: 0.960 ± 0.014
  ChestXRay_10_dual: 0.965 ± 0.014
  ChestXRay_10_dual_NN: 0.965 ± 0.013
  ChestXRay_10_dual_NN_no_raster: 0.962 ± 0.014
  ChestXRay_10_dual_no_raster: 0.961 ± 0.014
  ChestXRay_10_no_raster: 0.958 ± 0.014
  HybridGNet_ISBI: 0.963 ± 0.015

HD:
  ChestXRay_10: 37.493 ± 23.345
  ChestXRay_10_NN: 37.995 ± 21.243
  ChestXRay_10_NN_no_raster: 42.576 ± 20.179
  ChestXRay_10_dual: 33.193 ± 21.122
  ChestXRay_10_dual_NN: 35.920 ± 20.260
  ChestXRay_10_dual_NN_no_raster: 40.390 ± 19.358
  ChestXRay_10_dual_no_raster: 39.878 ± 20.396
  ChestXRay_10_no_raster: 41.570 ± 21.720
  HybridGNet_ISBI: 32.493 ± 23.394

Dataset: Montgomery, Organ: Lungs
DC:
  ChestXRay_10: 0.968 ± 0.018
  ChestXRay_10_NN: 0.974 ± 0.015
  ChestXRay_10_NN_no_raster: 0.966 ± 0.020
  ChestXRay_10_dual: 0.974 ± 0.019
  ChestXRay_10_dual_NN: 0.975 ± 0.018
  ChestXRay_10_dual_NN_no_raster:

In [9]:
import pandas as pd
import os

def create_multicolumn_results_table(results_path="../Results/CXRAY"):
    """
    Create a table with separate Organ and Dataset columns, models as multi-level columns (DC/HD)
    Rows are sorted by organ first, then dataset, so all organ results are grouped together
    """
    df = pd.DataFrame()
    models = sorted(os.listdir(results_path))
    
    # Load and process data
    for folder in models:
        if os.path.isdir(os.path.join(results_path, folder)):
            df_temp = pd.read_csv(os.path.join(results_path, folder, "results.csv"))
            df_temp["model"] = folder
            for i, row in df_temp.iterrows():
                image = row["image"]
                if image.startswith('MCU'):
                    df_temp.at[i, "dataset"] = "Montgomery"
                elif image.startswith('JPC'):
                    df_temp.at[i, "dataset"] = "JSRT"
                elif image.startswith('CHN'):
                    df_temp.at[i, "dataset"] = "Shenzhen"
                else:
                    df_temp.at[i, "dataset"] = "Padchest"
            df = pd.concat([df, df_temp])
    
    # Define custom sorting orders
    organ_order = ['Lungs', 'Heart', 'Clavicles']
    dataset_order = ['Shenzhen', 'Montgomery', 'Padchest', 'JSRT']
    
    # Include nnUNet dataframe
    path = "/home/ngaggion/Documents/Mask2Graph/Results/cxray_nnUNet_results.csv"
    df_nnunet = pd.read_csv(path)
    df_nnunet["model"] = "nnUNet"
    df = pd.concat([df, df_nnunet])
    
    # Get unique combinations and models, sorted by custom order
    organs = [organ for organ in organ_order if organ in df['organ'].unique()]
    datasets = [dataset for dataset in dataset_order if dataset in df['dataset'].unique()]
    models = sorted(df['model'].unique())
    
    # Create table data with organ-dataset combinations sorted properly
    table_data = []
    for organ in organs:
        for dataset in datasets:
            df_subset = df[(df['organ'] == organ) & (df['dataset'] == dataset)]
            if len(df_subset) > 0: # Only add rows where we have data
                row = {'Organ': organ, 'Dataset': dataset}
                for model in models:
                    df_model = df_subset[df_subset['model'] == model]
                    if len(df_model) > 0:
                        dc_mean = df_model['dc'].mean()
                        dc_std = df_model['dc'].std()
                        row[f"{model}_DC"] = f"{dc_mean:.3f} ± {dc_std:.3f}"
                    else:
                        row[f"{model}_DC"] = "N/A"
                table_data.append(row)
    
    # Create DataFrame
    result_df = pd.DataFrame(table_data)
    
    # Create multi-level column headers
    new_columns = [("", "Organ"), ("", "Dataset")]
    for model in models:
        new_columns.extend([
            (model, "DC Mean ± STD")
        ])
    
    # Reorder columns
    columns = ["Organ", "Dataset"]
    for model in models:
        columns.extend([f"{model}_DC",])
    
    result_df = result_df[columns]
    result_df.columns = pd.MultiIndex.from_tuples(new_columns)
    
    return result_df

# Main execution
if __name__ == "__main__":
    results_path = "../Results/CXRAY"
    
    # Create table with separate organ and dataset columns
    results_table = create_multicolumn_results_table(results_path)
    
    print("="*120)
    print("SEGMENTATION RESULTS BY ORGAN AND DATASET")
    print("="*120)
    
    print("\nPandas DataFrame:")
    print(results_table.to_string(index=False))

SEGMENTATION RESULTS BY ORGAN AND DATASET

Pandas DataFrame:
                      ChestXRay_10 ChestXRay_10_NN ChestXRay_10_NN_no_raster ChestXRay_10_dual ChestXRay_10_dual_NN ChestXRay_10_dual_NN_no_raster ChestXRay_10_dual_no_raster ChestXRay_10_no_raster HybridGNet_ISBI        nnUNet
    Organ    Dataset DC Mean ± STD   DC Mean ± STD             DC Mean ± STD     DC Mean ± STD        DC Mean ± STD                  DC Mean ± STD               DC Mean ± STD          DC Mean ± STD   DC Mean ± STD DC Mean ± STD
    Lungs   Shenzhen 0.961 ± 0.015   0.963 ± 0.014             0.960 ± 0.014     0.965 ± 0.014        0.965 ± 0.013                  0.962 ± 0.014               0.961 ± 0.014          0.958 ± 0.014   0.963 ± 0.015 0.969 ± 0.014
    Lungs Montgomery 0.968 ± 0.018   0.974 ± 0.015             0.966 ± 0.020     0.974 ± 0.019        0.975 ± 0.018                  0.972 ± 0.018               0.969 ± 0.019          0.964 ± 0.021   0.964 ± 0.031 0.982 ± 0.009
    Lungs   Padchest 0.953 

In [10]:
results_table.T

0              1  \
                               Organ                  Lungs          Lungs   
                               Dataset             Shenzhen     Montgomery   
ChestXRay_10                   DC Mean ± STD  0.961 ± 0.015  0.968 ± 0.018   
ChestXRay_10_NN                DC Mean ± STD  0.963 ± 0.014  0.974 ± 0.015   
ChestXRay_10_NN_no_raster      DC Mean ± STD  0.960 ± 0.014  0.966 ± 0.020   
ChestXRay_10_dual              DC Mean ± STD  0.965 ± 0.014  0.974 ± 0.019   
ChestXRay_10_dual_NN           DC Mean ± STD  0.965 ± 0.013  0.975 ± 0.018   
ChestXRay_10_dual_NN_no_raster DC Mean ± STD  0.962 ± 0.014  0.972 ± 0.018   
ChestXRay_10_dual_no_raster    DC Mean ± STD  0.961 ± 0.014  0.969 ± 0.019   
ChestXRay_10_no_raster         DC Mean ± STD  0.958 ± 0.014  0.964 ± 0.021   
HybridGNet_ISBI                DC Mean ± STD  0.963 ± 0.015  0.964 ± 0.031   
nnUNet                         DC Mean ± STD  0.969 ± 0.014  0.982 ± 0.009   

                                                          2              3  \
                               Organ                  Lungs          Lungs   
                               Dataset             Padchest           JSRT   
ChestXRay_10                   DC Mean ± STD  0.953 ± 0.016  0.967 ± 0.014   
ChestXRay_10_NN                DC Mean ± STD  0.954 ± 0.017  0.972 ± 0.011   
ChestXRay_10_NN_no_raster      DC Mean ± STD  0.948 ± 0.019  0.968 ± 0.012   
ChestXRay_10_dual              DC Mean ± STD  0.956 ± 0.022  0.975 ± 0.011   
ChestXRay_10_dual_NN           DC Mean ± STD  0.955 ± 0.020  0.975 ± 0.011   
ChestXRay_10_dual_NN_no_raster DC Mean ± STD  0.951 ± 0.018  0.971 ± 0.012   
ChestXRay_10_dual_no_raster    DC Mean ± STD  0.950 ± 0.020  0.970 ± 0.011   
ChestXRay_10_no_raster         DC Mean ± STD  0.946 ± 0.020  0.966 ± 0.012   
HybridGNet_ISBI                DC Mean ± STD  0.956 ± 0.023  0.973 ± 0.010   
nnUNet                         DC Mean ± STD  0.965 ± 0.017  0.979 ± 0.009   

                                                          4              5  \
                               Organ                  Heart          Heart   
                               Dataset             Padchest           JSRT   
ChestXRay_10                   DC Mean ± STD  0.925 ± 0.038  0.928 ± 0.038   
ChestXRay_10_NN                DC Mean ± STD  0.940 ± 0.022  0.935 ± 0.034   
ChestXRay_10_NN_no_raster      DC Mean ± STD  0.931 ± 0.028  0.936 ± 0.033   
ChestXRay_10_dual              DC Mean ± STD  0.939 ± 0.019  0.941 ± 0.027   
ChestXRay_10_dual_NN           DC Mean ± STD  0.942 ± 0.017  0.940 ± 0.027   
ChestXRay_10_dual_NN_no_raster DC Mean ± STD  0.939 ± 0.020  0.939 ± 0.029   
ChestXRay_10_dual_no_raster    DC Mean ± STD  0.930 ± 0.027  0.936 ± 0.032   
ChestXRay_10_no_raster         DC Mean ± STD  0.935 ± 0.029  0.935 ± 0.032   
HybridGNet_ISBI                DC Mean ± STD  0.939 ± 0.022  0.936 ± 0.030   
nnUNet                         DC Mean ± STD  0.947 ± 0.016  0.953 ± 0.025   

                                                          6  
                               Organ              Clavicles  
                               Dataset                 JSRT  
ChestXRay_10                   DC Mean ± STD  0.801 ± 0.067  
ChestXRay_10_NN                DC Mean ± STD  0.836 ± 0.068  
ChestXRay_10_NN_no_raster      DC Mean ± STD  0.827 ± 0.059  
ChestXRay_10_dual              DC Mean ± STD  0.860 ± 0.077  
ChestXRay_10_dual_NN           DC Mean ± STD  0.862 ± 0.078  
ChestXRay_10_dual_NN_no_raster DC Mean ± STD  0.854 ± 0.099  
ChestXRay_10_dual_no_raster    DC Mean ± STD  0.841 ± 0.062  
ChestXRay_10_no_raster         DC Mean ± STD  0.825 ± 0.078  
HybridGNet_ISBI                DC Mean ± STD  0.816 ± 0.068  
nnUNet                         DC Mean ± STD  0.951 ± 0.018

In [11]:
# provide the average DC and HD for each model across all datasets and organs

def average_metrics_across_models(df):
    """
    Calculate the average DC and HD for each model across all datasets and organs.
    Returns a DataFrame with models as rows and average metrics as columns.
    """
    models = df['model'].unique()
    results = []

    for model in models:
        df_model = df[df['model'] == model]
        dc_mean = df_model['dc'].mean()
        dc_std = df_model['dc'].std()
        hd_mean = df_model['hd'].mean()
        hd_std = df_model['hd'].std()
        
        results.append({
            'model': model,
            'DC Mean': f"{dc_mean:.3f} ± {dc_std:.3f}",
            'HD Mean': f"{hd_mean:.3f} ± {hd_std:.3f}"
        })
    
    return pd.DataFrame(results)
# Calculate average metrics across all models
average_results = average_metrics_across_models(df)
# Print the average results
print("\nAverage Metrics Across All Models:")
print(average_results.to_string(index=False))


Average Metrics Across All Models:
                         model       DC Mean         HD Mean
                  ChestXRay_10 0.928 ± 0.067 38.560 ± 19.668
               ChestXRay_10_NN 0.938 ± 0.057 36.581 ± 17.202
     ChestXRay_10_NN_no_raster 0.933 ± 0.057 40.906 ± 18.289
             ChestXRay_10_dual 0.944 ± 0.052 33.845 ± 18.941
          ChestXRay_10_dual_NN 0.944 ± 0.052 36.449 ± 18.056
ChestXRay_10_dual_NN_no_raster 0.940 ± 0.058 39.636 ± 17.408
   ChestXRay_10_dual_no_raster 0.936 ± 0.054 38.318 ± 19.617
        ChestXRay_10_no_raster 0.932 ± 0.060 38.499 ± 18.726
               HybridGNet_ISBI 0.934 ± 0.063 31.412 ± 18.171
                        nnUNet 0.964 ± 0.020 29.455 ± 21.202
